# CS781 Phase 1 - Decoder-only legal LM (ALiBi, 1B-token, 2xT4, resumable)

**How this notebook runs**
- Use **Save Version -> Save & Run All (Commit)** with *GPU T4 x2*, so training keeps going even if the browser closes.
- Section 4-5 (download + tokenizer + tokenization) run once. Tokenizer, token files and checkpoints are written to `/kaggle/working`.
- Training (section 8) runs as a 2-GPU DDP job. After `SESSION_BUDGET_HOURS` it saves a checkpoint and stops cleanly; the evaluation / generation cells then skip themselves.
- **To continue in a new session:** new version -> *Add Input -> Notebook Output Files -> this notebook* -> Run All. The tokenized corpus, tokenizer and newest checkpoint are found automatically (nothing is re-downloaded or re-tokenized) and the data order resumes exactly where it stopped.
- When the single pass over the corpus is finished, sections 9-14 evaluate the best checkpoint, generate the test predictions and build the submission zip.

# 1. Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

# 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

# 3. Configuration

In [ ]:
import os
import gc
import math
import time
import glob
import json
import random
import zipfile
import numpy as np
import torch

# -----------------------------------------------------------------------------
# File path setup
# -----------------------------------------------------------------------------
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

DATA_DIR = "Phase_1_Pretraining"
TRAIN_DIR = os.path.join(DATA_DIR, "Train")
TEST_FILE = os.path.join(DATA_DIR, "Test", "test.json")

# Collect ALL training files inside the Train directory (not just Supreme Court)
if os.path.exists(TRAIN_DIR):
    TRAIN_FILES = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.json")))
else:
    # Fallback explicit list matching HF repository structure before download
    TRAIN_FILES = [
        os.path.join(TRAIN_DIR, "train_supreme_court.json"),
        os.path.join(TRAIN_DIR, "train_high_court.json"),
        os.path.join(TRAIN_DIR, "train_statutes.json"),
        os.path.join(TRAIN_DIR, "train_tribunals.json"),
    ]

print(f"Discovered {len(TRAIN_FILES)} training file(s):")
for f in TRAIN_FILES:
    print(f" - {f}")

# -----------------------------------------------------------------------------
# Reproducibility & device
# -----------------------------------------------------------------------------
SEED = 42

def set_seed(seed: int = 42):
    """Ensures reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = torch.cuda.is_available()

# -----------------------------------------------------------------------------
# Hyperparameters: Model 1 (Decoder-only transformer, ALiBi positions)
# -----------------------------------------------------------------------------
TOTAL_TARGET_TOKENS = 1_000_000_000  # 1 Billion-token budget: ONE pass over it, LR anneals to MIN_LR exactly at the end
VAL_RATIO = 0.01

VOCAB_SIZE = 32000
D_MODEL = 768                # these four values are what the model is actually built from
N_LAYER = 10                 # (the old N_EMBD=560 / N_LAYER=8 / N_HEAD=8 block was never used by the model)
N_HEAD = 12
DROPOUT = 0.1
MODEL_CONTEXT_LENGTH = 2048
TRAIN_BLOCK_SIZE = 2048      # tokens per training sample

MODEL_KWARGS = dict(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEAD, n_layers=N_LAYER,
                    max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT, bias=True)

MICRO_BATCH_SIZE = 4         # samples per GPU per forward/backward (try 8 after the first steps if memory allows;
                             # 2,048-token samples need ~2x the logit memory of the encoder-decoder's 1,024 decoder tokens)
GLOBAL_BATCH_SAMPLES = 64    # samples per optimizer step across ALL GPUs; grad-accumulation is derived from it
LEARNING_RATE = 6e-4
MIN_LR = 6e-5
WEIGHT_DECAY = 0.1
WARMUP_FRACTION = 0.05       # warm-up = 5% of the total steps ...
MAX_WARMUP_STEPS = 1000      # ... but never more than this
# NOTE: the number of training steps is no longer hard-coded. One pass over the tokenized corpus =
# len(train_dataset) // GLOBAL_BATCH_SAMPLES; it is computed in section 7.

# Session / checkpoint policy (Kaggle sessions are time-limited)
SESSION_BUDGET_HOURS = 8.0   # training saves a checkpoint and stops cleanly this many hours after the notebook
                             # started (keep >= 1h below your session cap; raise it if your sessions allow ~12h)
CKPT_EVERY = 250             # optimizer steps between resumable checkpoints
EVAL_EVERY = 500             # optimizer steps between validation runs
LOG_EVERY = 50
VAL_SAMPLES = 400            # fixed validation subset (samples of 2,048 tokens)
DATALOADER_WORKERS = 2       # per GPU process
BYTES_PER_TOKEN = 4.0        # refined from real data after the tokenizer is trained

# Generation parameters
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95

# -----------------------------------------------------------------------------
# Output & checkpoint directories
# -----------------------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
INPUT_ROOT = "/kaggle/input"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-gpt2-final")

BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
TOKENIZER_PATH = os.path.join(OUTPUT_DIR, "legal_bpe.json")
TOKENIZATION_MARKER = os.path.join(OUTPUT_DIR, "tokenization_complete.json")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "train_config.json")
STATUS_PATH = os.path.join(OUTPUT_DIR, "train_status.json")
LOG_PATH = os.path.join(OUTPUT_DIR, "train_log.jsonl")

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Cross-session helpers
#    Kaggle wipes /kaggle/working between sessions. To continue in a new session, attach the previous
#    notebook version's OUTPUT as an input; artifacts are then found automatically (no hard-coded paths).
# -----------------------------------------------------------------------------
def find_artifact(pattern):
    """First match of `pattern` in this session's working dir, else anywhere under /kaggle/input."""
    for root in (OUTPUT_DIR, INPUT_ROOT):
        hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
        if hits:
            return hits[0]
    return None

# Session clock (used for the clean-stop deadline of the training run)
try:
    import psutil
    SESSION_START = psutil.Process(os.getpid()).create_time()
except Exception:
    SESSION_START = time.time()

print("\nConfiguration & session helpers successfully loaded.")

# 4. BPE Tokenizer training

In [ ]:
import os
import glob
import json
from huggingface_hub import hf_hub_download

# -----------------------------------------------------------------------------
# Reuse the tokenized corpus + tokenizer from a previous session if they exist
# (in /kaggle/working or in an attached previous-notebook output).
# -----------------------------------------------------------------------------
_marker = find_artifact("tokenization_complete.json")
_prev_dir = os.path.dirname(_marker) if _marker else None
if (_prev_dir
        and os.path.exists(os.path.join(_prev_dir, "legal_bpe.json"))
        and glob.glob(os.path.join(_prev_dir, "train_part_*.bin"))
        and glob.glob(os.path.join(_prev_dir, "val_part_*.bin"))):
    NEED_TOKENIZE = False
    TOKENIZED_DIR = _prev_dir
else:
    NEED_TOKENIZE = True
    TOKENIZED_DIR = OUTPUT_DIR

train_file_paths = []
if NEED_TOKENIZE:
    print("Downloading training files from Hugging Face...")
    for rel_path in TRAIN_FILES:
        try:
            downloaded_path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                filename=rel_path,
                repo_type="dataset",
                token=HF_TOKEN
            )
            train_file_paths.append(downloaded_path)
            print(f"  [+] Downloaded: {rel_path}")
        except Exception as e:
            print(f"  [-] Failed to download {rel_path}: {e}")
else:
    print(f"Found a finished tokenized corpus + tokenizer in {TOKENIZED_DIR} -> skipping the training-file download.")

print("\nDownloading test file...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

if NEED_TOKENIZE:
    print(f"\nSuccessfully downloaded {len(train_file_paths)} training file(s):")
    for path in train_file_paths:
        print(f" - {path}")

print(f"\nTest file downloaded to:\n - {test_file_path}")

In [ ]:
!pip install -q ijson

import os
import gc
import ijson
import random
from tokenizers import ByteLevelBPETokenizer, Tokenizer

TARGET_VOCAB_SIZE = VOCAB_SIZE   # 32,000 (token ids must stay < 65,536 because the corpus is stored as uint16)

if not NEED_TOKENIZE:
    # ---- Reuse the tokenizer that produced the existing .bin files (must match them exactly) ----
    tokenizer = Tokenizer.from_file(os.path.join(TOKENIZED_DIR, "legal_bpe.json"))
    print(f"Loaded existing tokenizer (vocab {tokenizer.get_vocab_size():,}) from {TOKENIZED_DIR}")
else:
    # -------------------------------------------------------------------------
    # Extract Representative Sample Across All Files without loading full JSONs
    # -------------------------------------------------------------------------
    SAMPLE_TARGET_BYTES = 200 * 1024 * 1024  # 200 MB text sample
    bytes_per_file = SAMPLE_TARGET_BYTES // max(len(train_file_paths), 1)

    sampled_texts = []
    total_bytes_collected = 0

    print(f"Streaming ~{SAMPLE_TARGET_BYTES / (1024 * 1024):.0f} MB of raw text for tokenizer training...")

    random.seed(SEED)

    for file_path in train_file_paths:
        file_bytes = 0
        file_name = os.path.basename(file_path)

        with open(file_path, "r", encoding="utf-8") as f:
            # ijson.items streams array elements one at a time without loading full file
            for record in ijson.items(f, "item"):
                text = record.get("text", "")
                if not text:
                    continue

                # Random sampling filter so we don't only take text from file head
                if random.random() < 0.1:
                    sampled_texts.append(text)
                    text_bytes = len(text.encode("utf-8"))
                    file_bytes += text_bytes
                    total_bytes_collected += text_bytes

                    if file_bytes >= bytes_per_file:
                        break

        gc.collect()
        print(f"  [+] Sampled {file_bytes / (1024 * 1024):.2f} MB from {file_name}")

    print(f"\nCorpus Extraction Complete: {len(sampled_texts):,} docs | {total_bytes_collected / (1024 * 1024):.2f} MB total.")

    # -------------------------------------------------------------------------
    # Train Custom Byte-Level BPE Tokenizer
    # -------------------------------------------------------------------------
    print(f"\nTraining ByteLevelBPETokenizer (Target Vocab Size: {TARGET_VOCAB_SIZE:,})...")
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train_from_iterator(sampled_texts, vocab_size=TARGET_VOCAB_SIZE)
    assert tokenizer.get_vocab_size() <= VOCAB_SIZE, "tokenizer vocab is larger than the model's embedding table"

    # Save it: the .bin files are only meaningful together with THIS tokenizer, and the notebook may restart.
    tokenizer.save(TOKENIZER_PATH)
    print(f"Saved tokenizer -> {TOKENIZER_PATH}  (vocab {tokenizer.get_vocab_size():,})")

    # Measure real bytes/token on a random probe so the corpus sub-sampling rate below is accurate.
    probe = random.sample(sampled_texts, min(2000, len(sampled_texts)))
    probe_bytes = sum(len(t.encode("utf-8")) for t in probe)
    probe_tokens = sum(len(e.ids) for e in tokenizer.encode_batch(probe))
    BYTES_PER_TOKEN = probe_bytes / max(probe_tokens, 1)
    print(f"Measured {BYTES_PER_TOKEN:.2f} bytes/token on a {len(probe):,}-document probe.")

    # Clear sample text memory
    del sampled_texts, probe
    gc.collect()
    print("Tokenizer training complete. RAM cleared.")

# 5. Dataset Tokenization

In [ ]:
import os
import glob
import json
import time
import shutil
import numpy as np
import ijson

os.environ["TOKENIZERS_PARALLELISM"] = "true"   # encode_batch is multi-threaded in Rust -> no multiprocessing needed

# -----------------------------------------------------------------------------
# Streaming tokenization.
# The old version did json.load() of the 6.5 GB file inside mp.Pool workers (~2x file size in RAM per worker);
# an OOM-killed worker makes Pool.starmap wait forever. Here ONE process streams records with ijson
# (constant memory), tokenizes them in small batches with the multi-threaded Rust tokenizer, and writes
# the uint16 .bin files incrementally, with progress logging.
# -----------------------------------------------------------------------------
def tokenize_streaming(tokenizer, file_paths, out_dir, val_ratio=0.01, target_tokens=1_000_000_000,
                       keep_prob=1.0, docs_per_batch=1000, seed=42, log_every=20):
    os.makedirs(out_dir, exist_ok=True)

    # remove partial files from an earlier failed run (the Dataset globs them!)
    for p in glob.glob(os.path.join(out_dir, "train_part_*.bin")) + glob.glob(os.path.join(out_dir, "val_part_*.bin")):
        os.remove(p)

    rng = np.random.default_rng(seed)
    f_train = open(os.path.join(out_dir, "train_part_0.bin"), "wb")
    f_val = open(os.path.join(out_dir, "val_part_0.bin"), "wb")
    n_train = n_val = n_docs = n_batches = 0
    t0 = time.time()
    batch = []

    def flush():
        nonlocal n_train, n_val, n_docs, n_batches
        if not batch:
            return
        for enc in tokenizer.encode_batch(batch):
            arr = np.asarray(enc.ids, dtype=np.uint16)
            if rng.random() < val_ratio:
                f_val.write(arr.tobytes()); n_val += arr.size
            else:
                f_train.write(arr.tobytes()); n_train += arr.size
        n_docs += len(batch)
        n_batches += 1
        batch.clear()
        if n_batches % log_every == 0:
            el = time.time() - t0
            tot = n_train + n_val
            print(f"[tok] {n_docs:,} docs | {tot/1e6:,.1f}M tokens | {tot/el/1e3:,.0f}k tok/s | {el/60:.1f} min", flush=True)

    try:
        done = False
        for path in file_paths:
            print(f"[tok] streaming {os.path.basename(path)}", flush=True)
            with open(path, "rb") as f:                       # binary mode for ijson
                for rec in ijson.items(f, "item"):            # one record at a time
                    text = rec.get("text")
                    if not text:
                        continue
                    # uniform random sub-sampling spreads the token budget over the WHOLE corpus
                    # (instead of only taking the first N GB of each file)
                    if keep_prob < 1.0 and rng.random() >= keep_prob:
                        continue
                    batch.append(text)
                    if len(batch) >= docs_per_batch:
                        flush()
                        if n_train + n_val >= target_tokens:
                            done = True
                            break
            if done:
                break
        flush()
    finally:
        f_train.close(); f_val.close()

    print(f"[tok] DONE  train={n_train:,}  val={n_val:,}  total={n_train+n_val:,} tokens in {(time.time()-t0)/60:.1f} min")
    return n_train, n_val, n_docs


if NEED_TOKENIZE:
    print(f"ijson backend: {ijson.backend}  (should be a yajl2_c variant; 'python' would be slow)")
    est_total_tokens = sum(os.path.getsize(p) for p in train_file_paths) / BYTES_PER_TOKEN
    KEEP_PROB = min(1.0, 1.08 * TOTAL_TARGET_TOKENS / est_total_tokens)   # ~8% head-room, stop at the target anyway
    print(f"Estimated corpus size: {est_total_tokens/1e9:.2f}B tokens -> keeping {KEEP_PROB:.1%} of documents "
          f"to reach a {TOTAL_TARGET_TOKENS/1e9:.1f}B-token budget.")

    n_train, n_val, n_docs = tokenize_streaming(
        tokenizer, train_file_paths, OUTPUT_DIR,
        val_ratio=VAL_RATIO, target_tokens=TOTAL_TARGET_TOKENS, keep_prob=KEEP_PROB, seed=SEED,
    )
    with open(TOKENIZATION_MARKER, "w") as f:   # written LAST: its presence means "tokenization finished"
        json.dump({"train_tokens": n_train, "val_tokens": n_val, "docs": n_docs, "vocab_size": tokenizer.get_vocab_size(),
                   "bytes_per_token": BYTES_PER_TOKEN, "keep_prob": KEEP_PROB}, f)
    TOKENIZED_DIR = OUTPUT_DIR
else:
    with open(os.path.join(TOKENIZED_DIR, "tokenization_complete.json")) as f:
        _m = json.load(f)
    print(f"Reusing tokenized corpus from {TOKENIZED_DIR}: {_m['train_tokens']:,} train / {_m['val_tokens']:,} val tokens.")
    if os.path.abspath(TOKENIZED_DIR) != os.path.abspath(OUTPUT_DIR):
        # copy forward so THIS version's output is self-contained (the next session then only needs this one attached)
        for pat in ("train_part_*.bin", "val_part_*.bin", "legal_bpe.json", "tokenization_complete.json"):
            for src in glob.glob(os.path.join(TOKENIZED_DIR, pat)):
                dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
        TOKENIZED_DIR = OUTPUT_DIR
        print("Copied tokenized corpus + tokenizer into /kaggle/working so they persist with this version.")

# 6. Decoder-only Transformer Model (ALiBi)

In [ ]:
%%writefile /kaggle/working/cs781_phase1/model_dt.py
import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

NEG_INF = -65504.0   # most negative finite fp16 value: "masked" without risking NaN from -inf arithmetic


@dataclass
class LegalLMConfig:
    vocab_size: int = 32000      # Custom BPE tokenizer vocabulary
    d_model: int = 768           # Model hidden dimension
    n_heads: int = 12            # Number of attention heads
    n_layers: int = 10           # Stacked layer depth (~95M parameters)
    max_seq_len: int = 2048      # Maximum context window length
    dropout: float = 0.1         # Dropout probability
    bias: bool = True            # Enable bias in Linear layers


def compute_alibi_slopes(n_heads: int) -> torch.Tensor:
    """
    Geometric slopes for Attention with Linear Biases (ALiBi).
    Reference: Press et al., "Train Short, Test Long: Attention with Linear Biases"
    (matches the reference implementation, including for head counts that are not a power of two, e.g. 12).
    """
    n = 2 ** math.floor(math.log2(n_heads))
    base = 2 ** (-(2 ** -(math.log2(n) - 3)))
    slopes = [base * (base ** i) for i in range(n)]

    if n < n_heads:
        extra_base = 2 ** (-(2 ** -(math.log2(2 * n) - 3)))
        # reference: get_slopes(2n)[0::2][: n_heads - n]  ->  extra_base ** (2i + 1)
        extra_slopes = [extra_base ** (2 * i + 1) for i in range(n_heads - n)]
        slopes.extend(extra_slopes)

    return torch.tensor(slopes, dtype=torch.float32)


class ALiBiCausalSelfAttention(nn.Module):
    """
    Causal self-attention with ALiBi, computed with F.scaled_dot_product_attention (fused kernel):
    the (T x T) attention map is never materialised in Python, and the causal mask + ALiBi penalty are
    handed to the kernel as ONE additive bias that is built once per forward pass (see CustomLegalLM._alibi_bias).
    Supports a key/value cache for fast generation.
    """
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        assert config.d_model % config.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = config.n_heads
        self.d_head = config.d_model // config.n_heads
        self.dropout_p = config.dropout

        self.c_attn = nn.Linear(config.d_model, 3 * config.d_model, bias=config.bias)   # fused Q,K,V
        self.c_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, attn_bias, past_kv=None, use_cache=False):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)   # (B, H, T, dh)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        # attn_bias is a {dtype: tensor} holder: the fp16 copy is made ONCE per forward pass and shared by all layers
        # (so autograd keeps a single copy for backward instead of one per layer)
        bias = attn_bias.get(q.dtype)
        if bias is None:
            bias = attn_bias[q.dtype] = attn_bias[torch.float32].to(q.dtype)
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=bias,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y, present


class FeedForward(nn.Module):
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model, bias=config.bias),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = ALiBiCausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x, attn_bias, past_kv=None, use_cache=False):
        a, present = self.attn(self.ln1(x), attn_bias, past_kv, use_cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, present


class CustomLegalLM(nn.Module):
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.d_model),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)]),
            ln_f=nn.LayerNorm(config.d_model),
        ))
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: input embedding == output projection
        self.transformer.wte.weight = self.lm_head.weight

        # ALiBi slopes live on the model (not per layer); non-persistent -> not stored in checkpoints
        self.register_buffer("alibi_slopes", compute_alibi_slopes(config.n_heads).view(1, config.n_heads, 1, 1),
                             persistent=False)
        self._bias_key = None
        self._bias_val = None

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def get_num_params(self) -> int:
        """Trainable parameters (the tied embedding / output matrix is counted once)."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def _alibi_bias(self, T: int, S: int, device) -> dict:
        """
        Additive attention bias (returned as {dtype: tensor}, fp32 entry of shape (1, H, T, S)): -slope * distance for past keys,
        a large negative value for future keys (causal mask). The T queries are the LAST T of the S keys,
        so the same function serves training (T == S) and cached decoding (T == 1, S = cache length + 1).
        Cached for repeated identical shapes (every training step).
        """
        key = (T, S, str(device))
        if self._bias_key == key:
            return self._bias_val
        q_pos = torch.arange(S - T, S, device=device).view(T, 1)
        k_pos = torch.arange(S, device=device).view(1, S)
        dist = (q_pos - k_pos).view(1, 1, T, S)
        bias = -self.alibi_slopes.to(device).float() * dist.clamp(min=0).float()
        bias = bias.masked_fill(dist < 0, NEG_INF)
        self._bias_key, self._bias_val = key, {torch.float32: bias}
        return self._bias_val

    def _run(self, input_ids, past_kvs=None, use_cache=False):
        B, T = input_ids.size()
        past_len = 0 if past_kvs is None else past_kvs[0][0].size(2)
        bias = self._alibi_bias(T, past_len + T, input_ids.device)

        x = self.transformer.drop(self.transformer.wte(input_ids))   # ALiBi supplies the position information
        presents = []
        for i, block in enumerate(self.transformer.h):
            x, present = block(x, bias, None if past_kvs is None else past_kvs[i], use_cache)
            presents.append(present)
        return self.transformer.ln_f(x), presents

    def forward(self, input_ids, targets=None):
        x, _ = self._run(input_ids)
        if targets is not None:
            logits = self.lm_head(x)                                  # full-sequence logits for the loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])                      # inference: last position only
            loss = None
        return logits, loss

    def forward_cached(self, input_ids, past_kvs=None):
        """Incremental forward: returns (last-position logits (B,1,V), new key/value cache)."""
        x, presents = self._run(input_ids, past_kvs, use_cache=True)
        return self.lm_head(x[:, [-1], :]), presents

    @staticmethod
    def _sample(logits, temperature, top_p):
        """logits: (B, V) -> next token ids (B, 1) with temperature + nucleus (top-p) sampling."""
        if temperature is None or temperature <= 0:
            return logits.argmax(dim=-1, keepdim=True)
        probs = F.softmax(logits.float() / temperature, dim=-1)
        if top_p is not None and top_p < 1.0:
            sorted_p, sorted_i = torch.sort(probs, descending=True, dim=-1)
            cum = sorted_p.cumsum(dim=-1)
            sorted_p = sorted_p.masked_fill((cum - sorted_p) > top_p, 0.0)   # always keeps the top token
            sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True)
            return sorted_i.gather(-1, torch.multinomial(sorted_p, 1))
        return torch.multinomial(probs, 1)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=1.0, top_p=1.0, stop_fn=None, check_every=16):
        """
        Autoregressive generation with a KV cache (one prefill pass, then one token per step).
        input_ids: (B, T) with NO padding (use B=1 for prompts of different lengths).
        stop_fn(list_of_new_ids_of_first_sequence) -> bool lets the caller stop early (e.g. after 128 words).
        Returns the new token ids, shape (B, n_new).
        """
        self.eval()
        room = max(1, self.config.max_seq_len - max_new_tokens)
        input_ids = input_ids[:, -room:]                      # keep the END of an over-long prompt
        logits, past = self.forward_cached(input_ids, None)
        new_ids = []
        for t in range(max_new_tokens):
            nxt = self._sample(logits[:, -1, :], temperature, top_p)
            new_ids.append(nxt)
            if stop_fn is not None and (t + 1) % check_every == 0 and stop_fn(torch.cat(new_ids, 1)[0].tolist()):
                break
            if t + 1 < max_new_tokens:
                logits, past = self.forward_cached(nxt, past)
        return torch.cat(new_ids, dim=1)

In [ ]:
import sys
import importlib
sys.path.insert(0, OUTPUT_DIR)
import model_dt
importlib.reload(model_dt)
from model_dt import CustomLegalLM, LegalLMConfig

# Build once on the CPU just to report the size (the GPUs are not touched; training builds the real model per GPU).
_probe = CustomLegalLM(LegalLMConfig(**MODEL_KWARGS))
print(f"Model 1 (Decoder-only, ALiBi) defined. Parameters: {_probe.get_num_params() / 1e6:.1f}M")
print("ALiBi slopes per head:", [round(s, 4) for s in _probe.alibi_slopes.flatten().tolist()])
del _probe
gc.collect()

# 7. Dataloader

In [ ]:
%%writefile /kaggle/working/cs781_phase1/data_dt.py
import glob
import os
import numpy as np
import torch
from torch.utils.data import Dataset


class CausalMemmapDataset(Dataset):
    """
    Disk-backed uint16 token stream cut into fixed windows for next-token prediction.
    Sample i:  x = tokens[i*B : i*B + B],   y = tokens[i*B + 1 : i*B + B + 1]
    (windows do not overlap except for the single shifted target token; a window may straddle two .bin files).
    """
    def __init__(self, file_pattern: str, block_size: int):
        self.block_size = block_size
        self.bin_files = [f for f in sorted(glob.glob(file_pattern)) if os.path.getsize(f) > 0]
        self.mmaps = [np.memmap(f, dtype=np.uint16, mode="r") for f in self.bin_files]
        self.lengths = [len(m) for m in self.mmaps]
        self.total_tokens = sum(self.lengths)
        self.cum_lengths = np.cumsum([0] + self.lengths)
        self.num_samples = max(0, (self.total_tokens - 1) // block_size)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        start = int(idx) * self.block_size
        remaining = self.block_size + 1                       # +1 token for the shifted target
        file_idx = int(np.searchsorted(self.cum_lengths, start, side="right") - 1)
        pos, parts = start, []
        while remaining > 0 and file_idx < len(self.mmaps):
            local = pos - int(self.cum_lengths[file_idx])
            take = min(remaining, self.lengths[file_idx] - local)
            parts.append(self.mmaps[file_idx][local:local + take].astype(np.int64))
            pos += take
            remaining -= take
            file_idx += 1
        chunk = np.concatenate(parts)
        return torch.from_numpy(chunk[:-1]), torch.from_numpy(chunk[1:])


def rank_micro_batches(num_samples, world_size, rank, micro_bs, accum, seed, start_step=0, max_steps=None):
    """
    Deterministic, resumable data order (one pass, no repeats).
    Optimizer step s always consumes perm[s*G:(s+1)*G] with G = world*micro_bs*accum, so a resumed run
    (even on a different number of GPUs, if G is kept constant) continues exactly where it stopped.
    Returns (total_steps, list_of_index_lists) for this rank, starting at `start_step`.
    """
    G = world_size * micro_bs * accum
    total_steps = num_samples // G
    n_steps = total_steps if max_steps is None else min(total_steps, max_steps)
    perm = np.random.default_rng(seed).permutation(num_samples)[: total_steps * G]
    perm = perm.reshape(total_steps, accum, world_size, micro_bs)[:n_steps]
    mine = perm[start_step:, :, rank, :].reshape(-1, micro_bs)
    return total_steps, mine.tolist()


def val_micro_batches(num_samples, world_size, rank, micro_bs, n_total):
    """Fixed random subset of the validation set, split across ranks."""
    per = world_size * micro_bs
    n = (min(num_samples, n_total) // per) * per
    if n == 0:
        return []
    idx = np.random.default_rng(0).permutation(num_samples)[:n].reshape(-1, world_size, micro_bs)
    return idx[:, rank, :].tolist()

In [ ]:
import sys
sys.path.insert(0, OUTPUT_DIR)
import importlib
import data_dt
importlib.reload(data_dt)
from data_dt import CausalMemmapDataset

train_dataset = CausalMemmapDataset(os.path.join(TOKENIZED_DIR, "train_part_*.bin"), TRAIN_BLOCK_SIZE)
val_dataset = CausalMemmapDataset(os.path.join(TOKENIZED_DIR, "val_part_*.bin"), TRAIN_BLOCK_SIZE)

# One pass over the whole tokenized corpus; the cosine schedule is built on this number, so the learning rate
# anneals to MIN_LR exactly when the data runs out.
MAX_TRAIN_STEPS = len(train_dataset) // GLOBAL_BATCH_SAMPLES
WARMUP_STEPS = int(max(10, min(MAX_WARMUP_STEPS, WARMUP_FRACTION * MAX_TRAIN_STEPS)))
TOKENS_PER_STEP = GLOBAL_BATCH_SAMPLES * TRAIN_BLOCK_SIZE

print(f"Dataset Loaded: {len(train_dataset):,} Training Samples | {len(val_dataset):,} Val Samples.")
print(f"Train tokens: {train_dataset.total_tokens/1e6:,.1f}M  |  tokens/optimizer step: {TOKENS_PER_STEP:,}")
print(f"MAX_TRAIN_STEPS = {MAX_TRAIN_STEPS:,} (one full pass)  |  warm-up = {WARMUP_STEPS} steps")
print("Wall-clock is printed during training (s/step and ETA) - read it after the first ~50 steps before planning sessions.")

# 8. Training Loop

Training script (written to disk and launched with `torchrun` on both T4s; resumable, time-boxed).

In [ ]:
%%writefile /kaggle/working/cs781_phase1/train_ddp_dt.py
"""
Resumable DDP pre-training script for the decoder-only (ALiBi) legal LM.
Launched by the notebook with:  python -m torch.distributed.run --standalone --nproc_per_node=<GPUs> train_ddp_dt.py config.json
Works with 1 or 2 GPUs (global batch is fixed, grad-accumulation adapts) and on CPU (gloo) for smoke tests.
"""
import os, sys, json, time, math, glob, shutil, contextlib
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

cfg = json.load(open(sys.argv[1]))
sys.path.insert(0, cfg["output_dir"])
from model_dt import CustomLegalLM, LegalLMConfig
from data_dt import CausalMemmapDataset, rank_micro_batches, val_micro_batches

# ----------------------------------------------------------------------------- distributed setup
rank = int(os.environ.get("RANK", 0))
world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
use_cuda = torch.cuda.is_available()
if use_cuda:
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
else:
    device = torch.device("cpu")
if world > 1:
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
is_main = rank == 0
use_amp = use_cuda


def log(*a):
    if is_main:
        print(*a, flush=True)


torch.manual_seed(cfg["seed"])
np.random.seed(cfg["seed"])

# ----------------------------------------------------------------------------- hyper-parameters
micro_bs = cfg["micro_batch_size"]
gbs = cfg["global_batch_samples"]
assert gbs % (world * micro_bs) == 0, f"global batch {gbs} not divisible by world({world})*micro({micro_bs})"
accum = gbs // (world * micro_bs)
block = cfg["block_size"]
tokens_per_step = gbs * block

train_ds = CausalMemmapDataset(os.path.join(cfg["tokenized_dir"], "train_part_*.bin"), block)
val_ds = CausalMemmapDataset(os.path.join(cfg["tokenized_dir"], "val_part_*.bin"), block)
total_steps = len(train_ds) // gbs
max_steps = total_steps if cfg.get("max_train_steps") is None else min(total_steps, int(cfg["max_train_steps"]))
warmup = int(max(10, min(cfg["max_warmup_steps"], cfg["warmup_fraction"] * max_steps)))
lr_max, lr_min = cfg["learning_rate"], cfg["min_lr"]


def get_lr(step):
    if step < warmup:
        return lr_max * (step + 1) / warmup
    if step >= max_steps:
        return lr_min
    ratio = (step - warmup) / max(1, max_steps - warmup)
    return lr_min + 0.5 * (1.0 + math.cos(math.pi * ratio)) * (lr_max - lr_min)


ckpt_dir = cfg["checkpoint_dir"]
os.makedirs(ckpt_dir, exist_ok=True)
latest_path = os.path.join(ckpt_dir, "latest_checkpoint.pt")
best_path = os.path.join(ckpt_dir, "best_model.pt")
status_path = cfg["status_path"]
log_path = cfg["log_path"]

log(f"[setup] world={world} device={device} micro_bs={micro_bs} accum={accum} global_batch={gbs} samples "
    f"({tokens_per_step:,} tokens/step)")
log(f"[setup] train samples={len(train_ds):,} val samples={len(val_ds):,} -> {max_steps:,} optimizer steps "
    f"(one pass), warmup={warmup}")

# ----------------------------------------------------------------------------- model / optimizer
model = CustomLegalLM(LegalLMConfig(**cfg["model"])).to(device)
raw_model = model


def build_optimizer(m):
    """AdamW; weight decay only on >=2-D tensors (matrices), not on biases / LayerNorm gains."""
    decay, nodecay = [], []
    for _, p in m.named_parameters():
        if p.requires_grad:
            (decay if p.ndim >= 2 else nodecay).append(p)
    groups = [{"params": decay, "weight_decay": cfg["weight_decay"]}, {"params": nodecay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=lr_max, betas=(0.9, 0.95), eps=1e-8)


opt = build_optimizer(raw_model)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def pick_resume():
    """Newest checkpoint (highest step) in the working dir or in any attached input (previous notebook output)."""
    cands = []
    for root in cfg["resume_roots"]:
        cands += glob.glob(os.path.join(root, "**", "latest_checkpoint.pt"), recursive=True)
    best_p, best_s = None, -1
    for p in sorted(set(cands)):
        try:
            try:
                m = torch.load(p, map_location="cpu", mmap=True, weights_only=False)
            except TypeError:
                m = torch.load(p, map_location="cpu")
            s = int(m.get("step", -1))
            del m
        except Exception as e:  # corrupt / half-written file
            log(f"[resume] skipping unreadable checkpoint {p}: {e}")
            continue
        if s > best_s:
            best_p, best_s = p, s
    return best_p, best_s


global_step = 0
best_val = float("inf")
resume_path, resume_step = pick_resume()
if resume_path is not None:
    ck = torch.load(resume_path, map_location="cpu", weights_only=False)
    if int(ck.get("global_batch_samples", gbs)) != gbs:
        raise RuntimeError("checkpoint was written with a different GLOBAL_BATCH_SAMPLES - the data order would not "
                           "line up. Keep GLOBAL_BATCH_SAMPLES unchanged across sessions.")
    raw_model.load_state_dict(ck["model_state"])
    opt.load_state_dict(ck["optimizer_state"])
    if use_amp and ck.get("scaler_state"):
        scaler.load_state_dict(ck["scaler_state"])
    global_step = int(ck["step"])
    best_val = float(ck.get("best_val_loss", float("inf")))
    log(f"[resume] restored step {global_step:,} from {resume_path} (best val so far: {best_val:.4f})")
    del ck
    # carry the best-model file forward so this session's output is self-contained
    if is_main and not os.path.exists(best_path):
        prev_best = sorted(glob.glob(os.path.join(os.path.dirname(resume_path), "best_model.pt")))
        if prev_best:
            shutil.copy2(prev_best[0], best_path)
            log(f"[resume] copied {prev_best[0]} -> {best_path}")
else:
    log("[resume] no checkpoint found -> starting from scratch")

if world > 1:
    model = DDP(model, device_ids=[local_rank] if use_cuda else None)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"[model] {n_params/1e6:.1f}M parameters")


# ----------------------------------------------------------------------------- helpers
def write_status(finished):
    if not is_main:
        return
    tmp = status_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"finished": bool(finished), "step": global_step, "max_steps": max_steps,
                   "best_val_loss": None if best_val == float("inf") else best_val,
                   "updated": time.time()}, f)
    os.replace(tmp, status_path)


def append_log(rec):
    if not is_main:
        return
    with open(log_path, "a") as f:
        f.write(json.dumps(rec) + "\n")


def save_checkpoint():
    if not is_main:
        return
    ck = {"model_state": raw_model.state_dict(), "optimizer_state": opt.state_dict(),
          "scaler_state": scaler.state_dict() if use_amp else None,
          "step": global_step, "best_val_loss": best_val, "tokens_seen": global_step * tokens_per_step,
          "global_batch_samples": gbs, "config": cfg["model"]}
    tmp = latest_path + ".tmp"
    torch.save(ck, tmp)
    os.replace(tmp, latest_path)          # atomic: a session killed mid-write never leaves a corrupt checkpoint
    log(f"[ckpt] saved step {global_step:,} -> {latest_path}")


@torch.no_grad()
def evaluate():
    raw_model.eval()
    batches = val_micro_batches(len(val_ds), world, rank, micro_bs, cfg["val_samples"])
    tot = torch.zeros(2, device=device)
    if batches:
        for x, y in DataLoader(val_ds, batch_sampler=batches, num_workers=0):
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                _, l = raw_model(x, y)
            tot[0] += l.float()
            tot[1] += 1
    if world > 1:
        dist.all_reduce(tot)
    raw_model.train()
    return (tot[0] / tot[1]).item() if tot[1] > 0 else float("nan")


# ----------------------------------------------------------------------------- training loop
t_start = time.time()
if global_step >= max_steps:
    log("[train] checkpoint is already at the final step - nothing left to train.")
    write_status(True)
else:
    _, train_batches = rank_micro_batches(len(train_ds), world, rank, micro_bs, accum, cfg["seed"],
                                          start_step=global_step, max_steps=max_steps)
    loader = DataLoader(train_ds, batch_sampler=train_batches, num_workers=cfg["num_workers"], pin_memory=use_cuda)
    it = iter(loader)
    model.train()
    opt.zero_grad(set_to_none=True)
    t_last, steps_since_log = time.time(), 0
    finished = False
    log(f"[train] starting at step {global_step:,}/{max_steps:,}")

    while global_step < max_steps:
        lr = get_lr(global_step)
        for g in opt.param_groups:
            g["lr"] = lr

        loss_sum = torch.zeros((), device=device)
        for micro in range(accum):
            x, y = next(it)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync_ctx = contextlib.nullcontext() if (world == 1 or micro == accum - 1) else model.no_sync()
            with sync_ctx:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    _, loss = model(x, y)
                scaler.scale(loss / accum).backward()
            loss_sum += loss.detach().float()
            del loss, _

        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        global_step += 1
        steps_since_log += 1

        if global_step % cfg["log_every"] == 0 or global_step == 1:
            avg = loss_sum / accum
            if world > 1:
                dist.all_reduce(avg)
                avg = avg / world
            v = avg.item()
            if not math.isfinite(v):
                log(f"[train] NON-FINITE loss at step {global_step} - aborting (last good checkpoint is kept).")
                sys.exit(2)
            now = time.time()
            sps = (now - t_last) / max(1, steps_since_log)
            eta_h = sps * (max_steps - global_step) / 3600
            mem = f" | mem {torch.cuda.max_memory_allocated()/2**30:.1f}GB" if use_cuda else ""
            log(f"step {global_step:6d}/{max_steps} | loss {v:.4f} | lr {lr:.2e} | gnorm {gnorm.item():.2f} | "
                f"{sps:.2f}s/step | {tokens_per_step/sps/1e3:.1f}k tok/s | elapsed {(now-t_start)/3600:.2f}h | "
                f"ETA {eta_h:.2f}h{mem}")
            append_log({"step": global_step, "train_loss": v, "lr": lr})
            t_last, steps_since_log = now, 0

        stop = False
        if global_step % 10 == 0:
            flag = torch.tensor([1.0 if time.time() > cfg["deadline_epoch"] else 0.0], device=device)
            if world > 1:
                dist.all_reduce(flag, op=dist.ReduceOp.MAX)
            stop = flag.item() > 0

        do_eval = (global_step % cfg["eval_every"] == 0) or global_step == max_steps
        if do_eval:
            val = evaluate()
            ppl = math.exp(val) if val < 20 else float("inf")
            log(f"\n[val] step {global_step} | val loss {val:.4f} | ppl {ppl:.2f}\n")
            append_log({"step": global_step, "val_loss": val})
            if is_main and val == val and val < best_val:  # val == val -> not NaN
                best_val = val
                tmp = best_path + ".tmp"
                torch.save({"model_state": raw_model.state_dict(), "step": global_step, "best_val_loss": best_val,
                            "config": cfg["model"]}, tmp)
                os.replace(tmp, best_path)
                log(f"[val] new best -> {best_path}")
            if world > 1:  # all ranks must agree on best_val for the checkpoint they might write
                bv = torch.tensor([best_val if best_val != float("inf") else 1e30], device=device)
                dist.broadcast(bv, src=0)
                best_val = float(bv.item()) if bv.item() < 1e29 else float("inf")

        if do_eval or stop or global_step % cfg["ckpt_every"] == 0:
            save_checkpoint()
        if global_step == max_steps:
            finished = True
        if stop and not finished:
            log(f"[train] session time budget reached at step {global_step:,} - checkpoint saved, stopping cleanly.")
            break

    write_status(finished)
    log(f"[train] {'FINISHED' if finished else 'PAUSED'} at step {global_step:,}/{max_steps:,} | best val {best_val:.4f}")

if world > 1:
    dist.barrier()
    dist.destroy_process_group()

In [ ]:
import os
import sys
import json
import time
import shutil
import subprocess

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU found - enable 'GPU T4 x2' under Session options -> Accelerator."
assert GLOBAL_BATCH_SAMPLES % (NUM_GPUS * MICRO_BATCH_SIZE) == 0, \
    f"GLOBAL_BATCH_SAMPLES ({GLOBAL_BATCH_SAMPLES}) must be divisible by NUM_GPUS*MICRO_BATCH_SIZE ({NUM_GPUS * MICRO_BATCH_SIZE})"

DEADLINE_EPOCH = SESSION_START + SESSION_BUDGET_HOURS * 3600
remaining_h = (DEADLINE_EPOCH - time.time()) / 3600
print(f"{NUM_GPUS} GPU(s) | grad-accum per GPU = {GLOBAL_BATCH_SAMPLES // (NUM_GPUS * MICRO_BATCH_SIZE)} | "
      f"time left before the clean-stop deadline: {remaining_h:.1f} h")
if remaining_h < 0.25:
    print("WARNING: almost no time left in this session's budget - training will pause right away. "
          "Start a new session / raise SESSION_BUDGET_HOURS.")

# carry the loss history of earlier sessions forward so the final loss plot covers the whole run
if not os.path.exists(LOG_PATH):
    _prev_log = find_artifact("train_log.jsonl")
    if _prev_log:
        shutil.copy2(_prev_log, LOG_PATH)
        print(f"Copied earlier training log -> {LOG_PATH}")

train_cfg = dict(
    seed=SEED, output_dir=OUTPUT_DIR, tokenized_dir=TOKENIZED_DIR, checkpoint_dir=CHECKPOINT_DIR,
    status_path=STATUS_PATH, log_path=LOG_PATH, resume_roots=[OUTPUT_DIR, INPUT_ROOT],
    micro_batch_size=MICRO_BATCH_SIZE, global_batch_samples=GLOBAL_BATCH_SAMPLES, block_size=TRAIN_BLOCK_SIZE,
    learning_rate=LEARNING_RATE, min_lr=MIN_LR, weight_decay=WEIGHT_DECAY,
    warmup_fraction=WARMUP_FRACTION, max_warmup_steps=MAX_WARMUP_STEPS,
    max_train_steps=None,                      # None -> one full pass over the tokenized corpus
    num_workers=DATALOADER_WORKERS, log_every=LOG_EVERY, eval_every=EVAL_EVERY, ckpt_every=CKPT_EVERY,
    val_samples=VAL_SAMPLES, deadline_epoch=DEADLINE_EPOCH, model=MODEL_KWARGS,
)
with open(CONFIG_PATH, "w") as f:
    json.dump(train_cfg, f, indent=2)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
# env["NCCL_P2P_DISABLE"] = "1"   # uncomment ONLY if the run hangs before step 1 (NCCL peer-to-peer issue)

cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NUM_GPUS}",
       os.path.join(OUTPUT_DIR, "train_ddp_dt.py"), CONFIG_PATH]
print("Launching:", " ".join(cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Training process exited with code {rc} - see the log above. "
                       f"The last good checkpoint (if any) is in {CHECKPOINT_DIR}.")

status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(status.get("finished", False))
print("\nTRAINING FINISHED - run the evaluation cells below." if TRAINING_DONE else
      f"\nPAUSED at step {status.get('step')}/{status.get('max_steps')}: checkpoint saved. Evaluation is skipped; "
      "start the next session (attach this notebook's output as input) to continue.")

# 9. Perplexity

In [ ]:
import sys
sys.path.insert(0, OUTPUT_DIR)
from model_dt import CustomLegalLM, LegalLMConfig
from data_dt import CausalMemmapDataset, val_micro_batches
from torch.utils.data import DataLoader
from tokenizers import Tokenizer

# The tokenizer lives in memory only if cells 4-5 ran in this kernel; otherwise load the saved one.
if "tokenizer" not in globals():
    tokenizer = Tokenizer.from_file(find_artifact("legal_bpe.json"))

_status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(_status.get("finished", False))


@torch.no_grad()
def evaluate_val_ppl(model, dataset, max_samples=2000, micro_bs=4):
    """Mean next-token loss over a fixed random subset of the held-out validation tokens."""
    model.eval()
    batches = val_micro_batches(len(dataset), 1, 0, micro_bs, max_samples)
    tot, n = 0.0, 0
    for x, y in DataLoader(dataset, batch_sampler=batches, num_workers=0):
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            _, loss = model(x, y)
        tot += loss.item(); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def evaluate_test_ppl(model, test_texts, tokenizer, max_len=MODEL_CONTEXT_LENGTH):
    """Token-weighted perplexity over the test texts (each truncated to the model context)."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for text in test_texts:
        if not text:
            continue
        tokens = tokenizer.encode(text).ids[:max_len]
        if len(tokens) < 4:
            continue
        x = torch.tensor([tokens[:-1]], dtype=torch.long, device=device)
        y = torch.tensor([tokens[1:]], dtype=torch.long, device=device)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            _, loss = model(x, y)
        total_loss += loss.item() * y.size(1)
        total_tokens += y.size(1)
    if total_tokens == 0:
        return float("inf"), float("inf")
    avg = total_loss / total_tokens
    return avg, (math.exp(avg) if avg < 20 else float("inf"))


def test_record_text(rec):
    return rec if isinstance(rec, str) else rec.get("text", rec.get("prompt", rec.get("input", "")))


def load_test_items(path):
    """test.json may be a dict {id: record} or a list of records -> list of (id, text)."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        return [(str(k), test_record_text(v)) for k, v in data.items()]
    return [(str(r.get("id", i)) if isinstance(r, dict) else str(i), test_record_text(r)) for i, r in enumerate(data)]


if not TRAINING_DONE:
    print(f"Pre-training is not finished yet (step {_status.get('step')}/{_status.get('max_steps')}). "
          "Skipping evaluation - continue training in the next session (attach this notebook's output as input).")
else:
    # Load best weights (fall back to the latest checkpoint if best_model.pt is not in this session's output)
    _best = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else (find_artifact("best_model.pt") or find_artifact("latest_checkpoint.pt"))
    assert _best is not None, "No trained weights found."
    ckpt = torch.load(_best, map_location="cpu", weights_only=False)
    model = CustomLegalLM(LegalLMConfig(**MODEL_KWARGS)).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded weights from {_best} (step {ckpt.get('step', 0)})")
    del ckpt

    val_dataset_eval = CausalMemmapDataset(os.path.join(TOKENIZED_DIR, "val_part_*.bin"), TRAIN_BLOCK_SIZE)
    val_loss = evaluate_val_ppl(model, val_dataset_eval)
    val_ppl = math.exp(val_loss)
    test_items = load_test_items(test_file_path)
    test_loss, test_ppl = evaluate_test_ppl(model, [t for _, t in test_items], tokenizer)

    print("\n" + "=" * 50)
    print(f"  Model 1 Validation Loss / PPL : {val_loss:.4f} / {val_ppl:.2f}")
    print(f"  Model 1 Test Loss / PPL       : {test_loss:.4f} / {test_ppl:.2f}")
    print("=" * 50 + "\n")
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
        json.dump({"val_loss": val_loss, "val_ppl": val_ppl, "test_loss": test_loss, "test_ppl": test_ppl}, f, indent=2)

# 10. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

# Reads the history the training script appended (it also covers earlier sessions, and works for a paused run too).
if not os.path.exists(LOG_PATH):
    print("No training log yet - nothing to plot.")
else:
    train_pts, val_pts = {}, {}
    with open(LOG_PATH) as f:
        for line in f:
            r = json.loads(line)
            if "train_loss" in r:
                train_pts[r["step"]] = r["train_loss"]      # later entries overwrite steps repeated after a resume
            if "val_loss" in r:
                val_pts[r["step"]] = r["val_loss"]
    t_steps, t_losses = zip(*sorted(train_pts.items())) if train_pts else ((), ())
    v_steps, v_losses = zip(*sorted(val_pts.items())) if val_pts else ((), ())

    plt.figure(figsize=(10, 5))
    plt.plot(t_steps, t_losses, label="Training Loss", alpha=0.7)
    plt.plot(v_steps, v_losses, label="Validation Loss", marker="o")
    plt.xlabel("Optimizer Steps")
    plt.ylabel("Cross-Entropy Loss")
    plt.title("Phase 1 Training and Validation Loss (Decoder-only)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=200)
    plt.show()
    print("Loss plot saved to:", LOSS_PLOT_PATH)

# 11. Save Final Model to Kaggle Output

In [ ]:
import shutil

if not TRAINING_DONE:
    print("Pre-training not finished - final model is saved once training completes.")
else:
    torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin"))
    with open(os.path.join(MODEL_OUTPUT_DIR, "config.json"), "w") as f:
        json.dump(MODEL_KWARGS, f, indent=2)
    shutil.copy2(find_artifact("legal_bpe.json"), os.path.join(MODEL_OUTPUT_DIR, "legal_bpe.json"))
    print("Final model (weights + config + tokenizer) saved to:", MODEL_OUTPUT_DIR)

# 12. Load Test Prompts

In [ ]:
test_items = load_test_items(test_file_path)
print("Test file:", test_file_path)
print("Number of test examples:", len(test_items))

# 13. Generate 128 Words (words, not tokens)

In [ ]:
def generate_completion(model, tokenizer, prompt, target_words=TARGET_GENERATION_WORDS):
    """
    Continues `prompt` with the trained decoder (KV-cached, temperature + top-p sampling) and returns exactly the
    first `target_words` generated words. Over-long prompts keep their END (the part that is being continued).
    """
    ids = tokenizer.encode(prompt).ids
    max_prompt = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
    ids = ids[-max_prompt:]
    if not ids:
        return ""
    x = torch.tensor([ids], dtype=torch.long, device=device)

    def enough_words(new_ids):                        # stop early once more than `target_words` words exist
        return len(tokenizer.decode(new_ids).split()) > target_words

    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        out = model.generate(x, max_new_tokens=MAX_GENERATION_TOKENS, temperature=TEMPERATURE,
                             top_p=TOP_P, stop_fn=enough_words, check_every=16)
    text = tokenizer.decode(out[0].tolist())
    return " ".join(text.split()[:target_words])

# 14. Generate Predictions for the Test Set

In [ ]:
if not TRAINING_DONE:
    print("Pre-training not finished - predictions are generated once training completes.")
else:
    predictions = {}
    total_examples = len(test_items)
    print(f"Total test examples: {total_examples}")
    t0 = time.time()

    for n, (example_id, prompt) in enumerate(test_items, 1):
        predictions[example_id] = {"text": generate_completion(model, tokenizer, prompt)}
        if n % 25 == 0 or n == total_examples:
            el = time.time() - t0
            print(f"Completed {n}/{total_examples} ({100 * n / total_examples:.1f}%) | {el/60:.1f} min | "
                  f"ETA {el / n * (total_examples - n) / 60:.1f} min", flush=True)
        if n % 100 == 0:    # periodic partial dump so a crash does not lose everything
            with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
                json.dump(predictions, f, ensure_ascii=False)

# 15. Save prediction.json

In [ ]:
if not TRAINING_DONE:
    print("Nothing to save yet.")
else:
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print("prediction.json saved:", PREDICTION_PATH)

# 16. Create CodaBench Submission ZIP

In [ ]:
if not os.path.exists(PREDICTION_PATH) or not TRAINING_DONE:
    print(f"No final prediction file yet - it is written once pre-training has finished.")
else:
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(PREDICTION_PATH, arcname="prediction.json")
    print("Codabench submission created:", SUBMISSION_ZIP, f"({os.path.getsize(SUBMISSION_ZIP) / 1024**2:.2f} MB)")

#### ---------------------------------COMPLETE-----------------------------------------